# Connectome Hypothesis Testing: EM1–EM4 (Fast Run)

**Evaluates Missed Synapses (EM1), False Synapses (EM2), Synapse Noise (EM3), and Split Errors (EM4)** on 1 null graph.

---
### Instructions for Kaggle:
1. In **Cell 3**, review the runtime parameters (already configured for `null_only` with 1 null graph).
2. Click **Run All**.
3. Download the generated results zip from the last cell.


In [ ]:
# Cell 1: Environment Setup & sys.path
# ============================================================
import os
import sys
from pathlib import Path

IS_KAGGLE = os.path.exists('/kaggle/input')

# Adjust codebase and dataset paths for Kaggle
KAGGLE_CODEBASE_PATH = Path('/kaggle/input/datasets/jeet7771/flywire-codebase')
KAGGLE_DATA_PATH     = Path('/kaggle/input/datasets/jeet7771/flywire-all-datasets')

if IS_KAGGLE:
    found_codebase = False
    possible_code_paths = [
        KAGGLE_CODEBASE_PATH,
        Path('/kaggle/working'),
        Path('/kaggle/input/flywire-codebase'),
        Path('/kaggle/input/flywire-hypothesis-kaggle-package'),
    ]
    for p in possible_code_paths:
        if (p / 'hypothesis_testing').exists() or p.exists():
            if str(p) not in sys.path:
                sys.path.insert(0, str(p))
            print(f'[OK] Codebase path added: {p}')
            found_codebase = True
            break
    if not KAGGLE_DATA_PATH.exists():
        print(f'[WARN] Data path not at default: {KAGGLE_DATA_PATH}')
    else:
        print(f'[OK] Data path found: {KAGGLE_DATA_PATH}')
else:
    REPO_ROOT = Path(os.getcwd()).resolve()
    if str(REPO_ROOT) not in sys.path:
        sys.path.insert(0, str(REPO_ROOT))
    print(f'[OK] Running locally. Repo root: {REPO_ROOT}')

print('Environment: KAGGLE' if IS_KAGGLE else 'Environment: LOCAL')


In [ ]:
# Cell 2: Framework Imports
# ============================================================
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

from hypothesis_testing.config import HypothesisExperimentConfig, ExecutionMode
from hypothesis_testing.runners.hypothesis_experiment_runner import HypothesisExperimentRunner

print('[OK] Hypothesis-testing framework imported successfully.')


In [ ]:
# ============================================================
# Cell 3: RUNTIME CONFIGURATION  <-- EDIT THIS CELL AS NEEDED
# ============================================================

# Execution mode:
#   - 'null_only' : Runs ONLY on Null graphs (bypasses duplicate Real runs)
#   - 'full'      : Runs on BOTH Real connectome and Null graphs
EXECUTION_MODE = 'null_only'

# Target connectome dataset: 'BANC' | 'FAFB' | 'MANC' | 'MAOL' | 'MCNS' | 'TEST'
DATASET_NAME = 'BANC'
LOCAL_DATASET_ROOT = 'research_data/raw'
NULL_MODEL_NAME = 'degree_preserving'

# Number of independently generated null graphs (1 topology):
NULL_GRAPH_SEEDS = [1]

ERROR_MODELS = [
    "missed_synapses",
    "false_synapses",
    "synapse_count_measurement",
    "split_errors"
]

# Error rates (10 rates)
ERROR_RATES = [0.000, 0.005, 0.010, 0.020, 0.030, 0.050, 0.075, 0.100, 0.150, 0.200]

# Perturbation trial seeds (5 replicates on the null graph)
RANDOM_SEEDS = [1, 2, 3, 4, 5]

ANALYSES = [
    'basic_structure',
    'degree_distribution',
    'connected_components',
    'reciprocity',
    'pagerank',
]

OUTPUT_ROOT = Path('results') / 'hypothesis_testing'

print(f'Execution Mode    : {EXECUTION_MODE}')
print(f'Dataset Name      : {DATASET_NAME}')
print(f'Error Models      : {ERROR_MODELS}')
print(f'Null Graph Seeds  : {NULL_GRAPH_SEEDS}')
print(f'Perturbation Seeds: {RANDOM_SEEDS}')


In [ ]:
# Cell 4: Resolve Dataset Path & Assemble Config
# ============================================================
if IS_KAGGLE:
    DATASET_ROOT = str(KAGGLE_DATA_PATH)
    CONFIGS_ROOT = str(KAGGLE_CODEBASE_PATH / 'configs') if (KAGGLE_CODEBASE_PATH / 'configs').exists() else 'configs'
else:
    DATASET_ROOT = '0-demodata' if DATASET_NAME.upper() == 'TEST' else LOCAL_DATASET_ROOT
    CONFIGS_ROOT = 'configs'

exp_config = HypothesisExperimentConfig(
    dataset_name=DATASET_NAME,
    dataset_root=DATASET_ROOT,
    configs_root=CONFIGS_ROOT,
    execution_mode=EXECUTION_MODE,
    null_model_name=NULL_MODEL_NAME,
    null_graph_seeds=NULL_GRAPH_SEEDS,
    error_model_names=ERROR_MODELS,
    error_rates=ERROR_RATES,
    random_seeds=RANDOM_SEEDS,
    analysis_names=ANALYSES,
    output_root=str(OUTPUT_ROOT),
)

print(f'[OK] Experiment configuration prepared for mode: {exp_config.execution_mode.value}.')


In [ ]:
# Cell 5: Execute Hypothesis Testing Pipeline
# ============================================================
runner = HypothesisExperimentRunner()
result = runner.run(exp_config)

print(f'Execution Status: {result.status}')
print(f'Total Runtime   : {result.runtime_seconds:.2f} seconds')
print(f'Deliverables    : {list(result.exported_paths.keys())}')


In [ ]:
# Cell 6: Inspect Generated Replicate-Level Outputs
# ============================================================
null_csv = OUTPUT_ROOT / DATASET_NAME / 'null_observations' / 'replicate_level_effects.csv'
if null_csv.exists():
    df_null = pd.read_csv(null_csv)
    print(f'[OK] Replicate records exported: {len(df_null)} rows')
    display(df_null.head(10))
else:
    print('Replicate CSV not found at expected path.')


In [ ]:
# Cell 7: Package Deliverables for 1-Click Download
# ============================================================
import shutil
out_folder = OUTPUT_ROOT / DATASET_NAME
zip_name = f'hypothesis_testing_{DATASET_NAME}_null_results'
if out_folder.exists():
    shutil.make_archive(zip_name, 'zip', str(out_folder))
    print(f'[OK] Created downloadable archive: {zip_name}.zip')
else:
    print('Output directory not found for archiving.')
